# 对象与属性

学习目标：能创建和检查对象属性，选择合适的复制和限制修改策略，并识别动态键与宿主复制的边界。

前置知识：对象引用、函数与闭包、可选链、循环和数组基本索引。

适用版本：ECMAScript 2025（ECMA-262 第 16 版）、Node.js 24.11.0；.mjs 文件按 ES 模块运行并采用严格模式。console 是宿主输出 API。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/09-objects-and-properties/。

1. [main.mjs](scripts/09-objects-and-properties/main.mjs)：按正文顺序运行全部正常示例。
2. [readonly-property.mjs](scripts/09-objects-and-properties/readonly-property.mjs)：只给 value 的新描述符默认不可写，严格模式下赋值失败。
3. [mixed-descriptor.mjs](scripts/09-objects-and-properties/mixed-descriptor.mjs)：同一描述符不能同时是数据属性和访问器属性。
4. [sealed-addition.mjs](scripts/09-objects-and-properties/sealed-addition.mjs)：封闭对象不能新增属性，即使已有属性仍可写。
5. [clone-function.mjs](scripts/09-objects-and-properties/clone-function.mjs)：复制图中只要包含不可克隆的函数，也会整体失败。
6. [clone-symbol.mjs](scripts/09-objects-and-properties/clone-symbol.mjs)：Symbol 值不可克隆，应与普通对象中被忽略的 Symbol 键区分。

Step 1：从项目根目录进入本章工作目录。

```bash
cd content/编程语言/javascript
```

Step 2：运行全部正常示例，按各片段中的输出注释核对。

```bash
node scripts/09-objects-and-properties/main.mjs
```

下文正常片段依次对应 main.mjs 中的代码；每段给出自身输入与定义。错误文件仅在相应小节单独运行。

## 1 字面量、属性键和方法

对象通过属性把键和值联系起来。属性键只能是字符串或 Symbol；其他键会先转换为属性键，所以数值 2 与字符串 "2" 访问的是同一个属性。点号适合固定标识符名称，方括号适合变量键、Symbol 或含连字符的名称。

对象字面量中，变量名简写相当于同名键对应该变量的当前值；[expression] 计算属性名，expression 是用于产生属性键的表达式；method() 是简写方法定义。本例 describe 是固定说明方法，方法中的 this 与调用方式在专章展开。

```javascript
const title = "属性练习";
const dynamicKey = "unit-name";
const id = Symbol("id");
const lesson = {
  title,
  [dynamicKey]: "对象",
  2: "第二项",
  [id]: 17,
  describe() { return "一份课程记录"; },
};
console.log(lesson.title, lesson[dynamicKey], lesson[2] === lesson["2"]);
console.log(lesson[id], lesson.describe());
// 输出依次为：
// 属性练习 对象 true
// 17 一份课程记录
```

## 2 读取、修改、删除与存在性

读取不存在的属性通常得到 undefined，但存在的属性也可以保存 undefined，不能用“读取结果不是 undefined”来判断属性存在。Object.hasOwn() 专门判断是否为自有属性，与值是什么无关。

属性可写且对象允许新增时，赋值可以修改或建立属性。delete 删除可配置的自有属性，不会删除同名变量，也不会自动删除继承属性；对缺失属性执行删除仍可成功。删除失败在当前严格模式中会抛出 TypeError，描述符小节会说明原因。

```javascript
const record = { title: "初稿", pending: undefined };
record.title = "定稿";
record.pages = 4;
console.log(record.title, record.pages, record.missing);
console.log(Object.hasOwn(record, "pending"), Object.hasOwn(record, "missing"));
console.log(delete record.pages, delete record.absent, Object.hasOwn(record, "pages"));
// 输出依次为：
// 定稿 4 undefined
// true false
// true true false
```

## 3 解构与对象展开

解构按属性名读取值并建立变量，冒号可以重命名；默认值只在读取结果为 undefined 时使用。解构并不保证只读取自有属性，它会走一般属性访问；右侧为 null 或 undefined 时不能直接解构。

对象剩余属性 ...rest 收集未取走的自有可枚举属性；对象展开 ...source 将源的自有可枚举属性复制到新对象。两者都包含符合条件的 Symbol 键，并读取属性的值，不保留原描述符。后写的同名属性覆盖先写的值。对象展开与数组或实参展开语法相似，但不要求源对象可迭代。

```javascript
const source = { title: "JS", count: undefined, note: null, nested: { level: 1 } };
const { title: name, count = 0, note = "默认", ...rest } = source;
const updated = { ...source, title: "JavaScript" };
console.log(name, count, note, Object.keys(rest).join(","));
console.log(updated.title, source.title, updated.nested === source.nested);
const inherited = Object.create({ mode: "继承值" });
const { mode } = inherited;
console.log(mode, Object.hasOwn({ ...inherited }, "mode"));
// 输出依次为：
// JS 0 null nested
// JavaScript JS true
// 继承值 false
```

## 4 getter 与 setter

访问器属性（accessor property）通过 getter 处理读取、setter 处理赋值，看起来像普通属性访问，实际可以执行代码。getter 不接收参数，setter 接收所赋的一个值；只有 getter 的属性不能直接赋值。它们不一定对应一个真实存储字段。

下面把分数保存在闭包变量里，setter 在接受有限且处于 0–100 的 Number 后才更新。访问器可以具有副作用；复制或枚举值时也可能触发 getter，不能把读取属性一概当成无操作成本的读取。

```javascript
let storedScore = 0;
const scoreCard = {
  get score() { return storedScore; },
  set score(value) {
    if (!Number.isFinite(value) || value < 0 || value > 100) {
      throw new RangeError("score 必须是 0 到 100 的有限数值");
    }
    storedScore = value;
  },
};
scoreCard.score = 88;
console.log(scoreCard.score, storedScore);
const accessor = Object.getOwnPropertyDescriptor(scoreCard, "score");
console.log(typeof accessor.get, typeof accessor.set, Object.hasOwn(accessor, "value"));
// 输出依次为：
// 88 88
// function function false
```

## 5 自有、继承与可枚举性

自有属性直接属于对象；继承属性通过原型链查找。Object.create(proto) 创建以 proto 为原型的对象；Object.getPrototypeOf() 读取原型，这里只为区分属性来源，完整原型机制留在原型章节。

in 判断属性是否可从对象或原型链找到，Object.hasOwn() 只检查对象自身。可枚举性是单独的属性标记：不可枚举不代表不能访问，也不代表数据保密。for...in 访问自有及继承的可枚举字符串键；Object.keys() 只返回自有可枚举字符串键。

```javascript
const prototype = { role: "reader" };
const member = Object.create(prototype);
member.name = "Lin";
Object.defineProperty(member, "token", { value: "local", enumerable: false });
console.log(member.role, "role" in member, Object.hasOwn(member, "role"));
console.log(Object.keys(member).join(","), member.token, Object.hasOwn(member, "token"));
member.role = "editor";
console.log(member.role, prototype.role);
delete member.role;
console.log(member.role, Object.getPrototypeOf(member) === prototype);
// 输出依次为：
// reader true false
// name local true
// editor reader
// reader true
```

## 6 属性描述符

属性描述符（property descriptor）说明属性的值或访问器，以及修改规则。Object.getOwnPropertyDescriptor() 返回某个自有属性的描述符；Object.defineProperty() 新建或修改描述符。

| 描述符字段 | 中文名称／含义 | 适用属性 |
| --- | --- | --- |
| value | 保存的值 | 数据属性 |
| writable | 是否允许通过赋值改变值 | 数据属性 |
| get | 读取时调用的函数或 undefined | 访问器属性 |
| set | 赋值时调用的函数或 undefined | 访问器属性 |
| enumerable | 是否参与相应属性枚举 | 两类均适用 |
| configurable | 是否允许删除和多数描述符变更 | 两类均适用 |

新建属性时，defineProperty() 中省略的布尔字段默认为 false；普通字面量数据属性则默认三项为 true。修改已有属性时省略字段会保留已有设置。数据字段 value/writable 不能和访问器字段 get/set 混在同一个描述符里。

configurable 为 false 后，不能随意变成 true、删除或在数据与访问器之间转换。不可配置的数据属性若仍可写，可以改值并把 writable 改为 false；一旦不可写，就不能重新开放写入。不要把“不可配置”简单等同于“值永远不能变”。

```javascript
const locked = {};
Object.defineProperty(locked, "id", { value: 7 });
const lockedDescriptor = Object.getOwnPropertyDescriptor(locked, "id");
console.log(lockedDescriptor.value, lockedDescriptor.writable, lockedDescriptor.enumerable, lockedDescriptor.configurable);
const normal = { count: 1 };
const normalDescriptor = Object.getOwnPropertyDescriptor(normal, "count");
console.log(normalDescriptor.writable, normalDescriptor.enumerable, normalDescriptor.configurable);
Object.defineProperty(normal, "count", { configurable: false });
normal.count = 2;
Object.defineProperty(normal, "count", { writable: false });
console.log(normal.count, Object.getOwnPropertyDescriptor(normal, "count").writable);
// 输出依次为：
// 7 false false false
// true true true
// 2 false
```

## 7 键值枚举与 fromEntries

Object.keys()、Object.values()、Object.entries() 分别返回自有可枚举字符串键、值、键值对数组；不会包括 Symbol 键。Object.getOwnPropertySymbols() 可取得自有 Symbol 键，不受可枚举性筛选。

对于普通对象，数组索引形式的字符串键按数值升序排列，其他字符串键按创建顺序，Symbol 键另按创建顺序。不要把对象枚举当成任意业务排序。

Object.fromEntries() 从可迭代的键值对创建普通对象，允许字符串或 Symbol 键，同名键后值覆盖前值；它不是描述符、继承关系或对象身份的往返保存。entries() 会先读取值，getter 可能在此被调用。

```javascript
const metaKey = Symbol("meta");
const metrics = { b: 2, 2: "two", 1: "one", a: 1, [metaKey]: "symbol" };
console.log(Object.keys(metrics).join(","));
console.log(Object.values(metrics).join(","));
const pairs = Object.entries(metrics);
console.log(pairs[0][0], pairs[0][1], Object.getOwnPropertySymbols(metrics).length);
const rebuilt = Object.fromEntries([["count", 1], ["count", 3], [metaKey, "kept"]]);
console.log(rebuilt.count, rebuilt[metaKey]);
// 输出依次为：
// 1,2,b,a
// one,two,2,1
// 1 one 1
// 3 kept
```

## 8 Object.assign 与浅拷贝

判断复制是否隔离修改时，不能只比较外层对象；还要追踪嵌套属性指向哪里。

Object.assign(target, ...sources) 把源的自有可枚举属性值写入 target，并返回 target；字符串键和 Symbol 键都可以复制。后面的源覆盖前面的值，源 getter 与目标 setter 都可能运行，发生错误前的写入不会自动回滚。

浅拷贝（shallow copy）只复制一层属性值：外层是新对象，nested 的值仍指向原来的对象。需要隔离哪一层，就显式复制哪一层。对象展开创建数据属性，不调用新对象目标上的 setter；它与 Object.assign 都不能代替按描述符复制。

![浅拷贝复制外层，嵌套对象仍共享。对象展开与 Object.assign 的这一层引用关系相同。](image/illustration/09-01-shallow-copy-graph.svg)

图示说明：图限于本例的普通数据属性，不展开 getter、setter 或任意复杂值的克隆规则。

下面的两个 === 比较分别检查外层身份与 nested 身份；随后修改 count，验证图中的共享关系。

```javascript
const original = { nested: { count: 1 } };
const shallow = Object.assign({}, original);
shallow.nested.count = 2;
console.log(shallow !== original, shallow.nested === original.nested, original.nested.count);
const isolated = { ...original, nested: { ...original.nested } };
isolated.nested.count = 9;
console.log(original.nested.count, isolated.nested.count);
let reads = 0;
let writes = 0;
const input = { get count() { reads += 1; return 5; } };
const target = { set count(value) { writes += value; } };
console.log(Object.assign(target, input) === target, reads, writes);
const copied = { ...input };
console.log(copied.count, reads, Object.hasOwn(Object.getOwnPropertyDescriptor(copied, "count"), "value"));
// 输出依次为：
// true true 2
// 2 9
// true 1 5
// 5 2 true
```

## 9 seal 与 freeze 的限制

Object.seal() 禁止新增属性，并把已有自有属性设为不可配置；可写数据属性仍能改值。Object.freeze() 在此基础上再把数据属性设为不可写。它们均返回原对象，不生成副本，且只处理对象自身这一层。

freeze 不冻结嵌套对象，也不会删除访问器 setter；一个被冻结的访问器仍可能修改外部状态。当前严格模式中，对受限属性的非法赋值抛出 TypeError，不能把这些操作理解成所有环境都静默忽略。

```javascript
const sealed = { count: 1 };
Object.seal(sealed);
sealed.count = 2;
console.log(sealed.count, Object.isSealed(sealed), Object.isFrozen(sealed));
const frozen = { nested: { count: 1 } };
console.log(Object.freeze(frozen) === frozen);
frozen.nested.count = 3;
console.log(Object.isFrozen(frozen), Object.isFrozen(frozen.nested), frozen.nested.count);
let stored = 0;
const frozenAccessor = { get value() { return stored; }, set value(next) { stored = next; } };
Object.freeze(frozenAccessor);
frozenAccessor.value = 6;
console.log(frozenAccessor.value);
// 输出依次为：
// 2 true false
// true
// true false 3
// 6
```

## 10 动态键与原型污染边界

外部提供的动态键不应未经检查就用于递归读写普通对象。支持历史 \_\_proto\_\_ 访问器的环境中，给普通对象该名称赋对象值可能改变目标对象的原型；多层路径经 constructor、prototype 等键还可能到达共享原型。原型污染（prototype pollution）会让本该只影响数据的操作改变后续继承查找。

本例只改变新建局部对象的原型，不修改 Object.prototype 或其他共享原型。对象字面量的 \_\_proto\_\_: value 有特殊原型设置语法，而 ["\_\_proto\_\_"] 是普通计算属性名。Object.assign() 会触发目标的继承 setter，对象展开则创建自有数据属性；这不意味着展开可替代输入校验，后续消费者仍可能不安全地处理同一个键。

固定字段的业务输入应采用允许字段列表并校验值；作为任意键字典时可用 Object.create(null)，避免继承属性与 \_\_proto\_\_ setter，读取时配合 Object.hasOwn()。无原型对象也不是能过滤全部恶意业务输入的校验器；复杂键值关系可在集合章节使用 Map。

```javascript
const payload = { ["__proto__"]: { localFlag: true } };
const assigned = Object.assign({}, payload);
const spread = { ...payload };
console.log(assigned.localFlag, Object.hasOwn(assigned, "__proto__"));
console.log(Object.hasOwn(spread, "__proto__"), Object.getPrototypeOf(spread) === Object.prototype);
const dictionary = Object.create(null);
dictionary["__proto__"] = "普通数据";
console.log(dictionary["__proto__"], Object.getPrototypeOf(dictionary) === null);
const incoming = { title: "课程", ["__proto__"]: { localFlag: true } };
const accepted = Object.create(null);
for (const key of Object.keys(incoming)) {
  if (key === "title" && typeof incoming[key] === "string") accepted[key] = incoming[key];
}
console.log(Object.keys(accepted).join(","), Object.hasOwn(Object.prototype, "localFlag"));
// 输出依次为：
// true false
// true true
// 普通数据 true
// title false
```

## 11 structuredClone 的宿主归属与数据图

structuredClone() 是 WHATWG 定义、由 Node.js 等宿主提供的全局 API，不是 ECMA-262 的 Object 方法。Node.js 自 17.0.0 提供该接口，本章使用固定的 24.11.0。支持哪些平台对象还取决于宿主，不因为 Node 提供同名方法就能复制浏览器全部对象。

它可复制可序列化的数据图，包括常见普通对象、数组和 Date、Map、Set、ArrayBuffer 等受支持类型；对已支持的嵌套对象递归复制，并保留图内循环和重复引用的关系。它不是给所有对象都有效的“深拷贝函数”。函数、Symbol 值、Proxy、WeakMap 等不可克隆值会导致 DataCloneError；浏览器 DOM 节点也不能据此复制。

普通对象复制不保留原型链、属性描述符、访问器本身或类私有元素，只读取自有可枚举字符串键的值；getter 可以在复制时执行。下面自引用 self 和两个共享子对象都在副本内部重新建立，原对象不受副本修改影响。

```javascript
const child = { score: 1 };
const graph = { left: child, right: child };
graph.self = graph;
const clone = structuredClone(graph);
clone.left.score = 9;
console.log(clone !== graph, clone.self === clone, clone.left === clone.right);
console.log(clone.left !== child, child.score, clone.right.score);
let cloneReads = 0;
const special = Object.create({ inherited: 1 });
Object.defineProperty(special, "score", { enumerable: true, get() { cloneReads += 1; return 8; } });
special[Symbol("hidden-key")] = 3;
const plain = structuredClone(special);
const plainDescriptor = Object.getOwnPropertyDescriptor(plain, "score");
console.log(cloneReads, plain.score, plainDescriptor.writable, Object.hasOwn(plainDescriptor, "get"));
console.log(Object.getPrototypeOf(plain) === Object.prototype, Object.hasOwn(plain, "inherited"), Object.getOwnPropertySymbols(plain).length);
// 输出依次为：
// true true true
// true 1 9
// 1 8 true false
// true false 0
```

## 12 转移不是保留两份副本

structuredClone() 的 transfer 选项对可转移对象改变所有权，不是单纯复制。下面 ArrayBuffer 代表一段字节缓冲区，byteLength 是字节数；将它放入 transfer 数组后，原缓冲区被分离，不能继续当作原来的数据存储使用。

这与普通对象浅拷贝的共享引用不同，也与默认复制出独立缓冲区不同。二进制视图、分离和缓冲区的完整行为在二进制数据专题展开；本例仅使用 4 字节内存，不持有文件或服务资源。

```javascript
const buffer = new ArrayBuffer(4);
const moved = structuredClone(buffer, { transfer: [buffer] });
console.log(buffer.byteLength, moved.byteLength);
// 输出依次为：
// 0 4
```

## 13 独立观察错误边界

以下文件分别启动新进程；预期退出码为 1。先根据代码判断错误原因，再运行相应命令，核对错误名称及对应位置。错误消息全文由宿主决定。

只给 value 的新描述符默认不可写，严格模式下赋值失败。

```javascript
const record = {};
Object.defineProperty(record, "id", { value: 1 });
record.id = 2;
// 预期错误：TypeError；Cannot assign to read only property 'id'
```

Step 1：独立运行 scripts/09-objects-and-properties/readonly-property.mjs。

```bash
node scripts/09-objects-and-properties/readonly-property.mjs
```

同一描述符不能同时是数据属性和访问器属性。

```javascript
Object.defineProperty({}, "value", { value: 1, get() { return 2; } });
// 预期错误：TypeError；Invalid property descriptor
```

Step 2：独立运行 scripts/09-objects-and-properties/mixed-descriptor.mjs。

```bash
node scripts/09-objects-and-properties/mixed-descriptor.mjs
```

封闭对象不能新增属性，即使已有属性仍可写。

```javascript
const record = Object.seal({ count: 1 });
record.extra = 2;
// 预期错误：TypeError；object is not extensible
```

Step 3：独立运行 scripts/09-objects-and-properties/sealed-addition.mjs。

```bash
node scripts/09-objects-and-properties/sealed-addition.mjs
```

复制图中只要包含不可克隆的函数，也会整体失败。

```javascript
structuredClone({ callback() {} });
// 预期错误：DataCloneError；could not be cloned
```

Step 4：独立运行 scripts/09-objects-and-properties/clone-function.mjs。

```bash
node scripts/09-objects-and-properties/clone-function.mjs
```

Symbol 值不可克隆，应与普通对象中被忽略的 Symbol 键区分。

```javascript
structuredClone({ value: Symbol("id") });
// 预期错误：DataCloneError；could not be cloned
```

Step 5：独立运行 scripts/09-objects-and-properties/clone-symbol.mjs。

```bash
node scripts/09-objects-and-properties/clone-symbol.mjs
```

## 本章小结

- 属性来源、可枚举性和描述符分别回答“从哪里来”“如何枚举”“怎样修改”。
- 展开与 assign 复制值且共享嵌套引用；freeze 和 seal 都不自动递归。
- 动态键需要输入规则；无原型字典可隔离继承查找，但不能代替业务校验。
- structuredClone 由宿主提供，只支持规定的数据类型；转移会使原缓冲区失去数据。

## 练习

1. 定义一个可枚举但不可写的 id 属性；核对 Object.keys() 包含 id、描述符 writable 为 false，单独赋值应报错。
2. 复制含嵌套统计对象的记录，要求只修改副本；核对外层和统计对象身份都不同，原来的计数保持不变。
3. 从带继承属性、不可枚举属性和 Symbol 属性的对象收集 Object.entries()；核对只得到自有可枚举字符串键。
4. 为只允许 title 和 count 的对象建立输入规则：title 必须是字符串，count 必须是非负安全整数，拒绝未允许字段；核对 \_\_proto\_\_ 和 constructor 没有进入结果。
5. 克隆有循环和重复引用的数据图；核对副本的自引用指向副本、两个内部别名仍相等，而它们与原图的子对象不同。

## 参考与引用来源

- TC39（tc39.es）：[§13.2.5 对象字面量、计算键与展开](https://tc39.es/ecma262/2025/multipage/ecmascript-language-expressions.html#sec-object-initializer)、[§6.1.7 属性及特性](https://tc39.es/ecma262/2025/multipage/ecmascript-data-types-and-values.html#sec-property-attributes)、[§6.2.6 描述符](https://tc39.es/ecma262/2025/multipage/ecmascript-data-types-and-values.html#sec-property-descriptor-specification-type)、[§20.1.2 Object 静态方法（§20.1.2.1 Object.assign 复制属性值）](https://tc39.es/ecma262/2025/multipage/fundamental-objects.html#sec-properties-of-the-object-constructor)、[§20.1.3.8 历史原型访问器](https://tc39.es/ecma262/2025/multipage/fundamental-objects.html#sec-object.prototype.__proto__)、[§10.1.11.1 自有属性顺序](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-ordinaryownpropertykeys)、[§7.3.15 freeze 与 seal](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-setintegritylevel)、[§7.3.25 剩余与展开复制](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-copydataproperties)：ECMAScript 2025 的对象属性、枚举、复制与限制修改。
- Node.js：[24.11.0 structuredClone](https://nodejs.org/download/release/v24.11.0/docs/api/globals.html#structuredclonevalue-options)：API 宿主归属与引入版本。
- WHATWG（html.spec.whatwg.org）：[结构化序列化](https://html.spec.whatwg.org/multipage/structured-data.html#structuredserializeinternal)、[反序列化](https://html.spec.whatwg.org/multipage/structured-data.html#structureddeserialize)、[转移](https://html.spec.whatwg.org/multipage/structured-data.html#structuredserializewithtransfer)：类型限制、普通对象字段、数据图身份及 ArrayBuffer 分离。
- MDN：[原型污染](https://developer.mozilla.org/en-US/docs/Web/Security/Attacks/Prototype_pollution)、[结构化克隆算法](https://developer.mozilla.org/en-US/docs/Web/API/Web_Workers_API/Structured_clone_algorithm)：动态键防护及复制限制的用法对照。